# MCR Stage B — R_compute saturation analysis

**Hypothesis**: correct rollouts reach a stable L23 residual state (= "decided the answer") earlier than wrong rollouts, which wander until max_new.

**Method** — per rollout:
1. Let `a_final = mean(a^L23[-32:])` (the last 32 tokens' mean, i.e. the state the model ended in).
2. Compute `cos_sim(a^L23(t), a_final)` for every token `t`.
3. **SaturationPoint** = first `t` where `cos_sim >= 0.95` for 32 consecutive tokens.
4. **wasted_tokens** = `response_len − SaturationPoint`.

**Decision rules**:
- Mann-Whitney U test on `wasted_tokens(correct)` vs `wasted_tokens(wrong)`, one-sided (correct < wrong).
- Spearman ρ between per-rollout wasted_tokens and outcome.
- **SHIP R_compute** if: `p < 0.01` AND `median(wasted|correct) < median(wasted|wrong)` by ≥20%.
- **DROP R_compute** if: `p > 0.05` OR difference < 10%.

**Runs on**: CPU only. Downloads shards lazily from HF. ~10 min for full 200-question corpus.

**Output**: `mcr_b_compute_saturation_analysis.json` + plots.

## Cell 1 — Install + imports

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub==1.5.0', 'safetensors', 'scipy',
                'matplotlib', 'seaborn', 'tqdm'], check=True)

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import json, re
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from safetensors import safe_open
from huggingface_hub import hf_hub_download, list_repo_files
from scipy.stats import mannwhitneyu, spearmanr
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

## Cell 2 — CFG + list shards

In [ ]:
CFG = dict(
    hf_repo='caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    target_layer=23,                 # layer to analyze saturation on
    cos_threshold=0.95,              # cos-sim to final state to count as saturated
    confirm_window=32,               # must stay saturated for this many consecutive tokens
    final_window=32,                 # tokens at end used to define a_final
    min_response_len=80,             # skip rollouts too short to analyze
    decision_p_ship=0.01,
    decision_p_drop=0.05,
    decision_min_pct_gap=0.20,       # ship if median(wrong)-median(correct) ≥ 20% of median(wrong)
    decision_drop_pct_gap=0.10,
)
print(json.dumps(CFG, indent=2))

files = list_repo_files(CFG['hf_repo'], repo_type='dataset')
shards = sorted([f for f in files if re.match(r'shards/q\d{4}\.safetensors$', f)])
print(f'\n{len(shards)} shards in repo')
if len(shards) < 200:
    print(f'\u26a0\ufe0f  Stage B still in progress ({len(shards)}/200). Analysis will be partial but valid directionally.')

## Cell 3 — Saturation detector

Given the L23 activation trajectory of a rollout, compute the first token at which the state has converged to the final answer direction.

In [ ]:
def saturation_point(acts: torch.Tensor, cos_threshold: float, confirm_window: int, final_window: int) -> int:
    """
    acts: [L, d_model] fp16/fp32.
    Returns: first t in [0, L-confirm_window] such that cos_sim(acts[t'], a_final) ≥ cos_threshold
             for all t' in [t, t+confirm_window). If never, returns L (= no saturation).
    """
    L, d = acts.shape
    if L < final_window + confirm_window:
        return L
    a = acts.float()
    a_final = a[-final_window:].mean(dim=0)
    a_final = F.normalize(a_final, dim=0)
    a_n = F.normalize(a, dim=-1)
    cos = a_n @ a_final  # [L]
    above = (cos >= cos_threshold).numpy()
    # rolling: need `confirm_window` consecutive True
    for t in range(L - confirm_window):
        if above[t:t+confirm_window].all():
            return t
    return L

# Quick unit test
x = torch.randn(500, 64)
# Force last 200 tokens to be close to a common direction
target = torch.randn(64)
x[300:] = target.unsqueeze(0) + 0.01 * torch.randn(200, 64)
sp = saturation_point(x, 0.95, 32, 32)
print(f'Unit test: SaturationPoint={sp} (expected ≈ 300, tolerate 280-320)')
assert 280 <= sp <= 320, f'saturation detector broken: got {sp}'

## Cell 4 — Scan all shards, build per-rollout table

In [ ]:
rows = []  # per-rollout records
skipped = 0

for shard in tqdm(shards, desc='Analyzing shards'):
    local = hf_hub_download(CFG['hf_repo'], shard, repo_type='dataset', force_download=False)
    with safe_open(local, framework='pt') as f:
        meta = f.metadata() or {}
        offsets_all = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        acts_L = f.get_tensor(f'acts_L{CFG["target_layer"]}')
        offsets = offsets_all[f'L{CFG["target_layer"]}']
        q_global_idx = int(meta['question_global_idx'])
        discipline = meta.get('discipline', '')
        difficulty = meta.get('difficulty', '')
        for r_idx, r in enumerate(rollouts):
            L = r['response_len']
            if L < CFG['min_response_len']:
                skipped += 1
                continue
            rollout_acts = acts_L[offsets[r_idx]:offsets[r_idx+1]]  # [L, d_model]
            sp = saturation_point(
                rollout_acts,
                cos_threshold=CFG['cos_threshold'],
                confirm_window=CFG['confirm_window'],
                final_window=CFG['final_window'],
            )
            wasted = max(0, L - sp)
            rows.append({
                'shard': shard,
                'q_idx': q_global_idx,
                'rollout_id': r_idx,
                'discipline': discipline,
                'difficulty': difficulty,
                'response_len': L,
                'saturation_point': sp,
                'wasted_tokens': wasted,
                'efficiency': sp / L,          # 0 = saturate immediately, 1 = never
                'saturated': sp < L,
                'correct': bool(r['correct']),
                'pred': r['pred'],
            })

print(f'\nProcessed {len(rows)} rollouts ({skipped} skipped as too short)')
import pandas as pd
df = pd.DataFrame(rows)
print(f'\nSummary:')
print(df[['response_len', 'saturation_point', 'wasted_tokens', 'efficiency', 'saturated', 'correct']].describe().round(2))

## Cell 5 — Core hypothesis tests

In [ ]:
correct = df[df['correct']]
wrong = df[~df['correct']]
print(f'Correct rollouts: {len(correct)} ({len(correct)/len(df):.1%})')
print(f'Wrong   rollouts: {len(wrong)} ({len(wrong)/len(df):.1%})')
print()

# Saturation rates
sat_rate_correct = correct['saturated'].mean()
sat_rate_wrong = wrong['saturated'].mean()
print(f'Saturated before max_new:  correct={sat_rate_correct:.1%}  |  wrong={sat_rate_wrong:.1%}')

# Medians
med_sp_c = correct['saturation_point'].median()
med_sp_w = wrong['saturation_point'].median()
med_wt_c = correct['wasted_tokens'].median()
med_wt_w = wrong['wasted_tokens'].median()
print(f'\nSaturationPoint (median):  correct={med_sp_c:.0f}  |  wrong={med_sp_w:.0f}')
print(f'Wasted tokens (median):    correct={med_wt_c:.0f}  |  wrong={med_wt_w:.0f}')
gap_pct = (med_wt_w - med_wt_c) / max(med_wt_w, 1)
print(f'Median wasted-token gap:   {gap_pct:.1%} (correct uses less "wasted" compute)')
print()

# Mann-Whitney U (one-sided: wasted_correct < wasted_wrong)
try:
    u_stat, u_p = mannwhitneyu(correct['wasted_tokens'], wrong['wasted_tokens'], alternative='less')
    print(f'Mann-Whitney U (correct<wrong wasted):  U={u_stat:.0f}  p={u_p:.2e}')
except Exception as e:
    u_p = 1.0
    print(f'Mann-Whitney failed: {e}')

# Spearman ρ between per-rollout wasted_tokens and (1 - correct)
rho, rho_p = spearmanr(df['wasted_tokens'], df['correct'].astype(int))
print(f'Spearman ρ(wasted_tokens, correct):       ρ={rho:+.4f}  p={rho_p:.2e}  (expect negative)')

# Ancillary: correlation with response_len alone (length-penalty baseline for comparison)
rho_len, rho_len_p = spearmanr(df['response_len'], df['correct'].astype(int))
print(f'Spearman ρ(response_len, correct):        ρ={rho_len:+.4f}  p={rho_len_p:.2e}  (length-only baseline)')

# Does saturation beat raw length?
print(f'\n|ρ(wasted)| vs |ρ(length)|:  {abs(rho):.4f}  vs  {abs(rho_len):.4f}  → ' +
      ('saturation is a STRONGER predictor than raw length ✅' if abs(rho) > abs(rho_len) else
       'raw length is as strong or stronger ⚠\ufe0f — R_compute may not add over simple length penalty'))

## Cell 6 — Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (1) SaturationPoint distribution by outcome
ax = axes[0, 0]
for outcome, label, color in [(True, 'Correct', '#2ecc71'), (False, 'Wrong', '#e74c3c')]:
    sub = df[df['correct'] == outcome]['saturation_point']
    ax.hist(sub, bins=30, alpha=0.6, label=f'{label} (n={len(sub)}, med={sub.median():.0f})', color=color)
ax.set_xlabel('SaturationPoint (token)')
ax.set_ylabel('Count')
ax.set_title('When does the L23 state converge?')
ax.legend()

# (2) Wasted tokens distribution by outcome
ax = axes[0, 1]
for outcome, label, color in [(True, 'Correct', '#2ecc71'), (False, 'Wrong', '#e74c3c')]:
    sub = df[df['correct'] == outcome]['wasted_tokens']
    ax.hist(sub, bins=30, alpha=0.6, label=f'{label} (med={sub.median():.0f})', color=color)
ax.set_xlabel('Wasted tokens (response_len - SaturationPoint)')
ax.set_ylabel('Count')
ax.set_title('Compute wasted after convergence')
ax.legend()

# (3) Scatter: response_len vs SaturationPoint, colored by outcome
ax = axes[1, 0]
for outcome, label, color, marker in [(True, 'Correct', '#2ecc71', 'o'), (False, 'Wrong', '#e74c3c', 'x')]:
    sub = df[df['correct'] == outcome]
    ax.scatter(sub['saturation_point'], sub['response_len'], alpha=0.5, s=15,
               label=label, color=color, marker=marker)
# diagonal = never wasted
lim = max(df['response_len'].max(), df['saturation_point'].max())
ax.plot([0, lim], [0, lim], '--', color='gray', label='SP = L (no waste)')
ax.set_xlabel('SaturationPoint')
ax.set_ylabel('Response length')
ax.set_title('Length vs convergence (off-diagonal = wasted compute)')
ax.legend()

# (4) Efficiency (SP/L) by outcome
ax = axes[1, 1]
sns.boxplot(data=df, x='correct', y='efficiency', ax=ax,
            palette={True: '#2ecc71', False: '#e74c3c'})
ax.set_xlabel('Correct')
ax.set_ylabel('Efficiency = SaturationPoint / response_len')
ax.set_title('Lower = more wasted compute; 1.0 = never saturated')

plt.tight_layout()
plt.savefig('mcr_b_compute_saturation.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n\u2705 Saved: mcr_b_compute_saturation.png')

## Cell 7 — Verdict + save artifact

In [ ]:
# Decision logic
if u_p < CFG['decision_p_ship'] and gap_pct >= CFG['decision_min_pct_gap']:
    verdict = 'SHIP_R_COMPUTE'
    emoji = '\u2705'
    msg = (f'Correct rollouts waste {gap_pct:.0%} less compute than wrong rollouts '
           f'(p={u_p:.1e} < {CFG["decision_p_ship"]}). R_compute enters MCR as the 5th term.')
elif u_p > CFG['decision_p_drop'] or gap_pct < CFG['decision_drop_pct_gap']:
    verdict = 'DROP_R_COMPUTE'
    emoji = '\u274c'
    msg = (f'Insufficient signal (p={u_p:.2e}, gap={gap_pct:.0%}). '
           f'Saturation does not clearly separate correct from wrong at this threshold. Drop or re-tune.')
else:
    verdict = 'INCONCLUSIVE'
    emoji = '\u26a0\ufe0f'
    msg = (f'Borderline (p={u_p:.2e}, gap={gap_pct:.0%}). Consider sweeping cos_threshold '
           f'in {{0.90, 0.92, 0.95, 0.98}} or confirm_window in {{16, 32, 64}} before deciding.')

print(f'{"=" * 60}')
print(f'{emoji}  VERDICT: {verdict}')
print(f'{"=" * 60}')
print(msg)

# Also surface compute-reward-over-length-penalty
beats_length_penalty = abs(rho) > abs(rho_len)
print(f'\nDoes saturation beat naive length penalty?  {"YES \u2705" if beats_length_penalty else "NO \u26a0\ufe0f"}')
print(f'  |\u03c1(wasted,correct)|={abs(rho):.4f}  vs  |\u03c1(length,correct)|={abs(rho_len):.4f}')

# Per-discipline breakdown (does signal hold uniformly?)
print(f'\nPer-discipline gap (positive = saturation-based R_compute helps):')
for disc, sub in df.groupby('discipline'):
    if len(sub) < 20: continue
    sc = sub[sub['correct']]
    sw = sub[~sub['correct']]
    if len(sc) < 5 or len(sw) < 5: continue
    gap = (sw['wasted_tokens'].median() - sc['wasted_tokens'].median()) / max(sw['wasted_tokens'].median(), 1)
    print(f'  {disc:30s}  n_c={len(sc):3d}  n_w={len(sw):3d}  gap={gap:+.1%}')

# Save artifact
analysis = {
    'timestamp': __import__('time').strftime('%Y-%m-%d %H:%M:%S'),
    'hf_repo': CFG['hf_repo'],
    'target_layer': CFG['target_layer'],
    'cos_threshold': CFG['cos_threshold'],
    'confirm_window': CFG['confirm_window'],
    'final_window': CFG['final_window'],
    'n_rollouts_analyzed': int(len(df)),
    'n_correct': int(df['correct'].sum()),
    'n_wrong': int((~df['correct']).sum()),
    'saturation_rate_correct': float(sat_rate_correct),
    'saturation_rate_wrong': float(sat_rate_wrong),
    'median_saturation_point_correct': float(med_sp_c),
    'median_saturation_point_wrong': float(med_sp_w),
    'median_wasted_tokens_correct': float(med_wt_c),
    'median_wasted_tokens_wrong': float(med_wt_w),
    'median_wasted_gap_pct': float(gap_pct),
    'mann_whitney_u': float(u_stat),
    'mann_whitney_p': float(u_p),
    'spearman_rho_wasted': float(rho),
    'spearman_rho_wasted_p': float(rho_p),
    'spearman_rho_length_baseline': float(rho_len),
    'spearman_rho_length_baseline_p': float(rho_len_p),
    'beats_length_penalty': bool(beats_length_penalty),
    'verdict': verdict,
    'message': msg,
}

with open('mcr_b_compute_saturation_analysis.json', 'w') as f:
    json.dump(analysis, f, indent=2)
print(f'\n\u2705 Saved: mcr_b_compute_saturation_analysis.json')

# Also save per-rollout dataframe for downstream use
df.to_parquet('mcr_b_saturation_per_rollout.parquet')
print(f'\u2705 Saved: mcr_b_saturation_per_rollout.parquet (per-rollout table, {len(df)} rows)')

## Cell 8 (optional) — Threshold sweep if verdict was INCONCLUSIVE

Only run if Cell 7 printed INCONCLUSIVE. Re-scans all shards at several `cos_threshold` values and reports the verdict at each. Picks the setting that maximizes `|ρ(wasted, correct)|` and re-emits the decision.

In [ ]:
if verdict != 'INCONCLUSIVE':
    print(f'Verdict was {verdict} — no need to sweep.')
else:
    print('Running threshold sweep...')
    sweep_results = []
    for cos_th in [0.90, 0.92, 0.95, 0.97, 0.98]:
        for win in [16, 32, 64]:
            sub_rows = []
            for shard in tqdm(shards, desc=f'cos={cos_th} w={win}', leave=False):
                local = hf_hub_download(CFG['hf_repo'], shard, repo_type='dataset', force_download=False)
                with safe_open(local, framework='pt') as f:
                    meta = f.metadata() or {}
                    offsets_all = json.loads(meta['offsets'])
                    rollouts = json.loads(meta['rollouts'])
                    acts_L = f.get_tensor(f'acts_L{CFG["target_layer"]}')
                    offsets = offsets_all[f'L{CFG["target_layer"]}']
                    for r_idx, r in enumerate(rollouts):
                        L = r['response_len']
                        if L < CFG['min_response_len']: continue
                        rollout_acts = acts_L[offsets[r_idx]:offsets[r_idx+1]]
                        sp = saturation_point(rollout_acts, cos_threshold=cos_th, confirm_window=win, final_window=CFG['final_window'])
                        sub_rows.append({'wasted': max(0, L - sp), 'correct': bool(r['correct'])})
            sdf = pd.DataFrame(sub_rows)
            rho_s, _ = spearmanr(sdf['wasted'], sdf['correct'].astype(int))
            sweep_results.append({'cos_threshold': cos_th, 'confirm_window': win, 'rho': rho_s})
            print(f'  cos={cos_th:.2f} w={win:2d}  ρ={rho_s:+.4f}')
    best = min(sweep_results, key=lambda r: r['rho'])  # most negative = best
    print(f'\nBest setting: cos={best["cos_threshold"]} w={best["confirm_window"]} ρ={best["rho"]:+.4f}')

## Interpreting the verdict

**If SHIP_R_COMPUTE**: The 5th term enters MCR. `R_compute(rollout) = -λ_c · wasted_tokens(rollout) / max_new` with `λ_c` tuned in Stage G (suggest start at 0.3). Compute SaturationPoint online during GRPO rollouts (cheap — one extra hook at L23 + cos-sim against trailing mean).

**If DROP_R_COMPUTE**: Remove the term from the final reward formula. Paper still ships 4-term MCR; mention the negative result in Limitations.

**If INCONCLUSIVE**: Run Cell 8 sweep, pick the best (cos_threshold, confirm_window), and re-run Cell 5 with those values before deciding.